# Train on Kaggle — Hierarchical Plant Disease Classification

Template for running the experiments on a Kaggle Notebook (T4/P100). Steps:
1. Attach the dataset `abdallahalidev/plantvillage-dataset` (the `color` folder).
2. Attach this repo (as a Kaggle Dataset/utility, or `git clone` it) so `src/` is importable.
3. Run the persisted split is already in `data/splits/` — reuse it (do **not** regenerate).
4. Train flat baseline / species / disease heads, then evaluate the pipeline.

> Metrics policy: **macro F1 is primary** (dataset is imbalanced). Accuracy is reported only as a secondary figure.

## 0. Setup paths & install extras

In [ ]:
import sys, os
from pathlib import Path

# Adjust REPO_DIR to where the repo lives on Kaggle (clone or attached dataset).
REPO_DIR = Path('/kaggle/working/plant-disease-hierarchical')
if not REPO_DIR.exists():
    # Example: clone your repo (replace with your remote)
    # !git clone https://github.com/<you>/plant-disease-hierarchical.git {REPO_DIR}
    raise SystemExit('Point REPO_DIR at the repository before continuing.')

sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)

# timm is needed for ResNet/EfficientNet; usually preinstalled on Kaggle, else:
# !pip install -q timm albumentations
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Point configs at the Kaggle dataset mount

The PlantVillage color images mount at a path like
`/kaggle/input/plantvillage-dataset/plantvillage dataset/color`.
Override `data.data_root` via an env var the train scripts read, or edit the config.
Here we set an env override and pass it through.

In [ ]:
DATA_ROOT = '/kaggle/input/plantvillage-dataset/plantvillage dataset/color'
assert Path(DATA_ROOT).exists(), f'Fix DATA_ROOT: {DATA_ROOT}'

# The persisted split is versioned in the repo; sanity-check it is present.
from src.data.splits import load_split, LabelMaps
maps = LabelMaps.from_json(REPO_DIR / 'data/splits/label_maps.json')
print('species:', maps.num_species, '| classes:', maps.num_classes)
print('train rows:', len(load_split(REPO_DIR / 'data/splits', 'train')))

## 2. Helper to run a train script with the Kaggle data root

We edit each config's `data_root` in-memory by writing a small override config,
or simply pass `--device cuda` and rely on the config's relative path if the
dataset is symlinked into `data/raw/`. The cleanest way: symlink the mount.

In [ ]:
# Symlink the Kaggle dataset to the path the configs expect (data/raw/...).
target = REPO_DIR / 'data/raw/plantvillage dataset/color'
target.parent.mkdir(parents=True, exist_ok=True)
if not target.exists():
    os.symlink(DATA_ROOT, target)
print('linked:', target, '->', os.readlink(target) if target.is_symlink() else 'N/A')

## 3. Flat baselines

Run the simple CNN and the ResNet-50 flat reference. Each writes
`experiments/<name>/best.pth` (best by val macro F1) + `history.json`.

In [ ]:
!python scripts/train_flat.py --config configs/baseline_cnn_flat.yaml --device cuda

In [ ]:
!python scripts/train_flat.py --config configs/resnet50_flat.yaml --device cuda

## 4. Hierarchical: species head + per-species disease heads

In [ ]:
!python scripts/train_species.py --config configs/resnet50_hierarchical.yaml --device cuda

In [ ]:
# Trains the 9 non-trivial species heads in a loop (5 single-class species skipped).
!python scripts/train_disease.py --config configs/resnet50_hierarchical.yaml --device cuda

## 5. End-to-end evaluation + flat-vs-hierarchical comparison

Reports end-to-end macro/weighted F1, balanced accuracy, **error propagation**
(what share of end-to-end errors are caused by the species head), and the
flat-vs-hierarchical macro-F1 delta — the key result of the thesis.

In [ ]:
!python scripts/evaluate_pipeline.py \
    --config configs/resnet50_hierarchical.yaml \
    --flat-config configs/resnet50_flat.yaml \
    --device cuda

## 6. Download results

`experiments/<name>/` holds `best.pth`, `history.json`, `config.yaml`,
`test_metrics.json`. Add `/kaggle/working/plant-disease-hierarchical/experiments`
to the notebook output, or zip it:

In [ ]:
!cd {REPO_DIR} && zip -r /kaggle/working/experiments.zip experiments -x '*.pth' && echo 'zipped (checkpoints excluded; remove -x to include)'